# 1. The data

This notebook shows what the model reads, using a small **synthetic** package (random numbers, real layout). The real data is not in the repository (see `docs/DATA.md`). Nothing here is a result.

In [1]:
import tempfile
from pathlib import Path
import numpy as np
import torch
from deep_multimodal import constants as c
from deep_multimodal.synthetic import write_synthetic_package
from deep_multimodal.data import load_fold

folder = Path(tempfile.mkdtemp()) / 'package'
write_synthetic_package(folder)
sorted(p.name for p in folder.iterdir())

['dm_2022', 'dm_common', 'fc_lookup', 'fold_2022']

## One row

A row is one cell on one target day. The key holds the cell number and the day (days since 1970-01-01).

In [2]:
keys = np.load(folder / 'fold_2022' / 'val_key.npy')
print('scored rows:', len(keys))
print('first keys (cell, day):', keys[:3].tolist())

scored rows: 300
first keys (cell, day): [[5, 19104], [34, 19094], [9, 19070]]


## Fire history

Each row has up to 32 occupied entries from the 7 by 7 window of cells around the target cell over the previous 14 days. Each entry has 4 numbers and a position index (negative means an empty slot).

In [3]:
tokens = np.load(folder / 'fold_2022' / 'val_token_positions.npy')
print('channels per entry:', c.FIRE_CHANNELS)
print('occupied entries in the first 5 rows:', (tokens[:5] >= 0).sum(1).tolist())

channels per entry: ('detected', 'onset', 'log_count', 'log_power')
occupied entries in the first 5 rows: [13, 9, 14, 13, 14]


## The other inputs

Nine per-row columns (burn history, recent fire summaries, calendar), 21 static columns per cell, a 60-day weather sequence and a day-ahead forecast.

In [4]:
print(c.ROW_COLUMNS)
print(c.STATIC_COLUMNS[:6], '...', len(c.STATIC_COLUMNS), 'columns')
print(c.WEATHER_CHANNELS)

['bf_log_n10_nbhd', 'bf_years_since', 'fwi', 'fh_detected_ring1_last7d', 'fh_detected_ring3_last7d', 'fh_detected_days_own_last14d', 'fh_days_since_detection_own', 'sin_doy', 'cos_doy']
['dem_mean', 'dem_sd', 'dem_slope', 'dem_sin_aspect', 'dem_cos_aspect', 'dem_tpi'] ... 21 columns
['t2m', 'vpd', 'ws', 'rh', 'sw1', 'sw2', 'rain7', 'dsr', 'ffmc', 'dmc', 'dc', 'isi', 'bui', 'fwi', 'tmax09', 'sw1_anom', 'sw2_anom', 'rain', 'u', 'v']


## Loading a fold

`load_fold` splits the fit rows by year (the last fit year is used to stop training), standardises with fit-row statistics only, and builds each row's weather window from its cell and day.

In [5]:
fold = load_fold(folder, 2022, torch.device('cpu'))
print('fit rows:', fold.fit_all.size, '| stage 1 training:', fold.train.size, '| stopping rows:', fold.stop.size, '| scored rows:', fold.validation.size)
batch = fold.validation.batch(torch.arange(4))
{k: tuple(v.shape) for k, v in batch.items()}

fit rows: 1200 | stage 1 training: 561 | stopping rows: 639 | scored rows: 300


{'fire_features': (4, 32, 4),
 'fire_positions': (4, 32),
 'static': (4, 21),
 'row_values': (4, 9),
 'weather': (4, 60, 37),
 'forecast_hours': (4, 12, 5),
 'forecast_daily': (4, 20),
 'forecast_missing': (4,),
 'label': (4,)}

The weather window has 60 days and 37 numbers per day (20 channels and 17 anomalies). A day with no forecast is flagged and masked later, never filled with zeros.

In [6]:
print('forecast days missing among the first 300 scored rows:', int(fold.validation.forecast_missing.sum()))

forecast days missing among the first 300 scored rows: 0
